# SEACOFS Eddy Dataset Pipeline

This notebook is a concise control panel for the modular SEACOFS eddy dataset pipeline. The scientific functions live in `src/seacofs_eddy_dataset/`; this notebook only loads config, runs stages, and checks outputs.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PROJECT_ROOT

## Load Config

Copy `config/example.yaml` to `config/local.yaml` and edit paths before running the pipeline.

In [ ]:
from seacofs_eddy_dataset.config import load_config
from seacofs_eddy_dataset.pipeline import STAGES, run_stage, run_all

CONFIG_PATH = PROJECT_ROOT / "config" / "local.yaml"
config = load_config(CONFIG_PATH)

config.output_root

In [ ]:
[(stage.name, stage.description) for stage in STAGES]

## Run Individual Stages

Start with `detect_nencioli`. Later stages are scaffolded and will become runnable as they are implemented.

In [ ]:
run_stage("detect_nencioli", config)

## Run A Stage Sequence

Edit this list as stages become available.

In [ ]:
STAGES_TO_RUN = [
    "detect_nencioli",
    # "fit_doppio_surface",
    # "track_eddies",
    # "process_tracked_dataset",
    # "compute_vertical_profiles",
    # "qc_vertical_profiles",
    # "compute_tilt",
    # "analyse_tilt",
]

for stage_name in STAGES_TO_RUN:
    run_stage(stage_name, config)

## Inspect Detection Outputs

In [ ]:
import pandas as pd

detection_dir = config.output_root / "detections"
detection_files = sorted(detection_dir.glob("fnumber=*.parquet"))

len(detection_files), detection_files[:3]

In [ ]:
if detection_files:
    df_detect = pd.concat((pd.read_parquet(path) for path in detection_files), ignore_index=True)
    display(df_detect.head())
    display(df_detect.groupby("Cyc").size().rename("count"))
    display(df_detect.groupby("fnumber").size().rename("detections").describe())
else:
    print(f"No detection files found in {detection_dir}")

## Run Everything

Use this only once every stage has been implemented and tested individually.

In [ ]:
# run_all(config)